# Sagittal Landmark Detection — Colab Fine-tuning

既存の学習済みモデル (`best.pt`) を出発点に、追加データで継続学習する。

**事前準備:**
1. Colabメニューの「ランタイム → ランタイムのタイプを変更」で **GPU (T4)** を選択
2. Google Drive に以下を配置:
   - 追加学習データ (`*_image.npy` + `*_landmarks.json`) を任意フォルダに
   - 既存の `best.pt` を任意の場所に
3. 下の設定セルのパスを変更して全セルを実行

**新しいランドマーク種類を追加する場合:**
- `LANDMARKS` に追加したい点名をコンマ区切りで追加する（例: `"L1_ant,L1_post,L2_ant,L2_post,S1_ant,S1_post,FH"`）
- 学習データの `*_landmarks.json` に追加した点の座標 (`landmarks_ijk` 内) が必要
- 既存の5点は既存チェックポイントから自動移植され、新しい点だけランダム初期化される

In [ ]:
DRIVE_DATA_DIR   = "/content/drive/MyDrive/anglist_data"   # 追加学習データフォルダ
DRIVE_CHECKPOINT = "/content/drive/MyDrive/best.pt"        # 既存モデルのチェックポイント
EPOCHS     = 50
BATCH_SIZE = 4
RESIZE     = [512, 512]
SIGMA      = 15.0
LR         = 1e-4   # ファインチューニングは小さめの学習率を推奨
# ランドマーク種類（コンマ区切り）。新しい点を追加する場合はここに追記する
LANDMARKS  = "L1_ant,L1_post,S1_ant,S1_post,FH"

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# リポジトリのクローンと依存関係インストール
!git clone https://github.com/masaki39/anglist /content/anglist
!pip install -q tqdm onnxruntime onnxscript

In [ ]:
# データをローカルにコピー（Drive経由のI/Oより高速）
import shutil, os
LOCAL_DATA = "/content/data"
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA)
files = os.listdir(LOCAL_DATA)
print(f"データファイル数: {len(files)}")
print("\n".join(sorted(files)[:10]))

In [ ]:
# GPU確認
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ファインチューニング実行
!cd /content/anglist/train && python train.py \
    --data-dir {LOCAL_DATA} \
    --save-dir /content/runs \
    --checkpoint {DRIVE_CHECKPOINT} \
    --landmarks {LANDMARKS} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --sigma {SIGMA} \
    --resize {RESIZE[0]} {RESIZE[1]}

In [ ]:
# ONNX エクスポート（単一ファイルに統合）
import os
!pip install -q onnx
!cd /content/anglist/train && python export_onnx.py \
    --checkpoint /content/runs/best.pt \
    --output /content/model_raw.onnx \
    --height {RESIZE[0]} \
    --width {RESIZE[1]}

import onnx
model_proto = onnx.load("/content/model_raw.onnx", load_external_data=True)
onnx.checker.check_model(model_proto)
onnx.save_model(model_proto, "/content/model.onnx", save_as_external_data=False)
# .meta.json を同名ファイルにコピー
import shutil
if os.path.exists("/content/model_raw.onnx.meta.json"):
    shutil.copy("/content/model_raw.onnx.meta.json", "/content/model.onnx.meta.json")
print("model.onnx (single file) ready:", os.path.getsize("/content/model.onnx") // 1024, "KB")

In [ ]:
# モデルをダウンロード
from google.colab import files
files.download('/content/model.onnx')
if os.path.exists('/content/model.onnx.meta.json'):
    files.download('/content/model.onnx.meta.json')
files.download('/content/runs/best.pt')

In [ ]:
# (オプション) Google Driveにも保存
import shutil, os
shutil.copy('/content/model.onnx', f"{DRIVE_DATA_DIR}/../model_finetuned.onnx")
if os.path.exists('/content/model.onnx.meta.json'):
    shutil.copy('/content/model.onnx.meta.json', f"{DRIVE_DATA_DIR}/../model_finetuned.onnx.meta.json")
shutil.copy('/content/runs/best.pt', f"{DRIVE_DATA_DIR}/../best_finetuned.pt")
print("Drive に保存完了")